# Nemesis — Backtest Analysis

Evaluates two strategies over historical J-Quants Pro data:

1. **MultiFactorStrategy** — Normal market: TDnet events + supply/demand + US overnight + macro
2. **ShockRecoveryStrategy** — After US market drops: buy oversold JP stocks with no bad news

**Trading assumption:** Buy at T+1 open, sell at T+2 open (overnight hold)

**No look-ahead bias:** Signals use only data available BEFORE market open

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) != 'Nemesis' else os.getcwd())

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

from datetime import date
import pandas as pd
import numpy as np

print('✅ Setup complete')

In [ ]:
# ─── Backtest configuration ───────────────────────────────────────────
BT_START = date(2024, 1, 4)    # Start date
BT_END   = date(2024, 12, 30)  # End date
TOP_N    = 20                   # Portfolio size
SLIPPAGE_BPS = 5.0              # One-way slippage

print(f'Backtest: {BT_START} to {BT_END}, top_n={TOP_N}, slippage={SLIPPAGE_BPS}bps')

## Step 1: Load Historical Data

Fetches all daily quotes for the backtest period from J-Quants Pro.

In [ ]:
from japan_stock_daily.collectors.jquants_collector import JQuantsCollector
from datetime import timedelta

jquants = JQuantsCollector()

# Fetch all daily quotes for backtest period + 40 days buffer
# NOTE: This may take a few minutes for a 1-year range
print('Fetching historical quotes from J-Quants Pro...')
print('(This will take a few minutes for a 1-year range)')

all_rows = []
current = BT_START - timedelta(days=45)  # Buffer for 30-day lookback
end_with_buffer = BT_END + timedelta(days=5)

from tqdm.auto import tqdm
dates = []
d = current
while d <= end_with_buffer:
    if d.weekday() < 5:  # Weekdays only
        dates.append(d)
    d += timedelta(days=1)

for d in tqdm(dates, desc='Fetching daily quotes'):
    try:
        daily = jquants.get_daily_quotes(d)
        if daily is not None and not daily.empty:
            all_rows.append(daily)
    except Exception as e:
        pass  # Skip missing dates

all_quotes = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
print(f'\nLoaded {len(all_quotes):,} daily quote records across {all_quotes["code"].nunique():,} stocks')

In [ ]:
# Load supply/demand data for the period
print('Fetching margin and short selling data...')
margin_rows, short_rows, breakdown_rows = [], [], []

for d in tqdm(dates[::5], desc='Fetching supply/demand'):  # Every 5 days (weekly data)
    try:
        m = jquants.get_weekly_margin_interest(d)
        if m is not None and not m.empty:
            margin_rows.append(m)
    except Exception:
        pass
    try:
        s = jquants.get_short_selling_positions(d)
        if s is not None and not s.empty:
            short_rows.append(s)
    except Exception:
        pass

all_margin = pd.concat(margin_rows, ignore_index=True) if margin_rows else pd.DataFrame()
all_short  = pd.concat(short_rows, ignore_index=True) if short_rows else pd.DataFrame()
print(f'Margin records: {len(all_margin):,} | Short records: {len(all_short):,}')

## Step 2: Run Shock Recovery Backtest

In [ ]:
from backtest.engine import BacktestEngine
from backtest.strategies.shock_recovery import ShockRecoveryStrategy
from japan_stock_daily.collectors.us_overnight_collector import USOverNightCollector

# Build data loader that provides US overnight + supply/demand per date
us_collector = USOverNightCollector()
universe = jquants.get_listed_companies()

def data_loader(signal_date):
    us_data = us_collector.get_historical_overnight(signal_date)
    # Get latest margin/short data before signal_date
    margin_latest = all_margin[all_margin['date'].dt.date <= signal_date] if not all_margin.empty else pd.DataFrame()
    if not margin_latest.empty:
        margin_latest = margin_latest.sort_values('date').groupby('code').last().reset_index()
    short_latest = all_short[all_short['date'].dt.date <= signal_date] if not all_short.empty else pd.DataFrame()
    if not short_latest.empty:
        short_latest = short_latest.sort_values('date').groupby('code').last().reset_index()
    return {
        'us_overnight': us_data,
        'margin': margin_latest,
        'short': short_latest,
        'universe': universe,
        # TDnet/EDINET disabled in backtest for speed (re-enable for accuracy)
        'tdnet': pd.DataFrame(),
        'edinet': pd.DataFrame(),
    }

engine = BacktestEngine(all_quotes=all_quotes, slippage_bps=SLIPPAGE_BPS, data_loader=data_loader)

print('Running Shock Recovery Strategy backtest...')
shock_result = engine.run(
    ShockRecoveryStrategy(top_n=TOP_N, shock_threshold=0.03, volume_spike_min=1.8),
    start_date=BT_START,
    end_date=BT_END,
)
shock_result.print_summary()

## Step 3: Parameter Sweep — Optimize Shock Recovery

In [ ]:
print('Parameter sweep for ShockRecoveryStrategy (may take 10-15 minutes)...')

sweep_results = engine.sweep_parameters(
    strategy_class=ShockRecoveryStrategy,
    param_grid={
        'shock_threshold':  [0.02, 0.03, 0.04, 0.05],
        'volume_spike_min': [1.5, 2.0, 2.5, 3.0],
        'top_n':            [10, 20],
        'hold_days':        [1, 2, 3],
        'target_pct':       [0.02, 0.03, 0.04],
        'stop_pct':         [0.015, 0.02, 0.03],
    },
    start_date=BT_START,
    end_date=BT_END,
)

print('\nTop 10 parameter combinations by Sharpe Ratio:')
display_cols = ['shock_threshold', 'volume_spike_min', 'top_n',
                'sharpe_ratio', 'win_ratio', 'annual_return', 'max_drawdown', 'profit_factor']
sweep_results[[c for c in display_cols if c in sweep_results.columns]].head(10)

## Step 4: Equity Curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Equity curve
ax1 = axes[0]
cum_shock = shock_result.get_cum_returns()
ax1.plot(cum_shock.index, cum_shock.values, label='Shock Recovery', color='#e94560', linewidth=2)
ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax1.set_title('Cumulative Returns (Equity Curve)', fontsize=14)
ax1.set_ylabel('Portfolio Value (1.0 = Starting Capital)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Daily PnL distribution
ax2 = axes[1]
shock_pnl = shock_result.daily_pnl * 100
ax2.hist(shock_pnl[shock_pnl > 0], bins=30, alpha=0.6, color='green', label='Wins')
ax2.hist(shock_pnl[shock_pnl < 0], bins=30, alpha=0.6, color='red', label='Losses')
ax2.axvline(x=0, color='black', linewidth=1)
ax2.set_title('Daily Return Distribution (%)', fontsize=14)
ax2.set_xlabel('Daily Return (%)')
ax2.set_ylabel('Frequency')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reports/backtest_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/backtest_results.png')

## Step 5: Trade Log Analysis

In [ ]:
trade_log = shock_result.trade_log
print(f'Total trades: {len(trade_log)}')
print(f'Win ratio: {(trade_log["is_win"].sum() / len(trade_log) * 100):.1f}%')
print(f'Avg win: {trade_log[trade_log["is_win"]]["return_pct"].mean():.3f}%')
print(f'Avg loss: {trade_log[~trade_log["is_win"]]["return_pct"].mean():.3f}%')
print()

# Best trades
print('Top 10 Best Trades:')
trade_log.nlargest(10, 'return_pct')[[
    'signal_date', 'entry_date', 'code', 'entry_price', 'exit_price', 'return_pct'
]]


In [ ]:
# Monthly returns heatmap
from backtest.metrics import compute_monthly_returns

monthly = compute_monthly_returns(shock_result.daily_pnl)
monthly['monthly_pct'] = monthly['monthly_return'] * 100

print('Monthly Returns (%)')
print(monthly['monthly_pct'].round(2).to_string())

## Step 6: Calendar Analysis (Day-of-Week Effect)

In [ ]:
# Day-of-week performance
pnl_df = shock_result.daily_pnl.to_frame('return')
pnl_df.index = pd.to_datetime(pnl_df.index)
pnl_df['weekday'] = pnl_df.index.day_name()
pnl_df['return_pct'] = pnl_df['return'] * 100

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
day_stats = pnl_df.groupby('weekday')['return_pct'].agg(['mean', 'count', lambda x: (x > 0).mean() * 100])
day_stats.columns = ['avg_return', 'n_days', 'win_rate']
day_stats = day_stats.reindex([d for d in day_order if d in day_stats.index])

print('Day-of-Week Performance (Shock Recovery):')
day_stats